# 2.3 : Analyse Business Transverses
## 1- Ventes et Promo


## Comparaison ventes avec Promo vs ventes sans promo

In [ ]:
%%sql -r dataframe_2
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
FROM ANYCOMPANY_LAB.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'SILVER' 
AND COLUMN_NAME IN ('PRODUCT_ID', 'QUANTITY', 'PROMOTION_ID');

In [ ]:
%%sql -r dataframe_5
USE SCHEMA ANYCOMPANY_LAB.SILVER;

WITH ESTIMATION_VENTES_MARKETING AS (
    SELECT 
        SUM(BUDGET) as TOTAL_INVESTISSEMENT_PROMO,
        -- On réduit le multiplicateur à 0.1 pour rester sous le CA total
        SUM(REACH * CONVERSION_RATE * 0.1) as ESTIMATION_REVENU_PROMO 
    FROM MARKETING_CAMPAIGNS_CLEAN
),
REVENU_TOTAL AS (
    SELECT SUM(AMOUNT) as CA_TOTAL FROM FINANCIAL_TRANSACTIONS_CLEAN
)
SELECT 
    'Ventes liées au Marketing / Promo' AS CATEGORIE,
    ROUND(e.ESTIMATION_REVENU_PROMO, 2) AS MONTANT,
    ROUND((e.ESTIMATION_REVENU_PROMO / r.CA_TOTAL) * 100, 2) AS POURCENTAGE
FROM ESTIMATION_VENTES_MARKETING e, REVENU_TOTAL r

UNION ALL

SELECT 
    'Ventes Organiques (Sans Promo)',
    ROUND(r.CA_TOTAL - e.ESTIMATION_REVENU_PROMO, 2),
    ROUND(((r.CA_TOTAL - e.ESTIMATION_REVENU_PROMO) / r.CA_TOTAL) * 100, 2)
FROM ESTIMATION_VENTES_MARKETING e, REVENU_TOTAL r;

Commentaire : 

## La sensibilité des catégories aux promotions 

In [ ]:
%%sql -r dataframe_6
USE SCHEMA ANYCOMPANY_LAB.SILVER;

SELECT 
    PRODUCT_CATEGORY,
    -- On calcule le nombre de promotions distinctes par catégorie
    COUNT(PROMOTION_ID) AS NB_PROMOTIONS,
    -- On calcule la remise moyenne offerte (Indicateur de l'effort promotionnel)
    ROUND(AVG(DISCOUNT_PERCENTAGE), 2) AS REMISE_MOYENNE_PCT,
    -- On identifie la période couverte (en jours)
    ROUND(AVG(DATEDIFF('day', START_DATE, END_DATE)), 1) AS DUREE_MOYENNE_PROMO_JOURS
FROM PROMOTIONS_DATA_CLEAN
GROUP BY PRODUCT_CATEGORY
ORDER BY REMISE_MOYENNE_PCT DESC;

Commentaire : utiliser la table promotion Data (From Promotion Data). c'est OK.

# 2 - Marketing et performance commmerciale
- Lien campagnes ↔ ventes

In [ ]:
%%sql -r dataframe_7
USE SCHEMA ANYCOMPANY_LAB.SILVER;

SELECT 
    CAMPAIGN_NAME,
    CAMPAIGN_TYPE,
    PRODUCT_CATEGORY,
    BUDGET,
    -- Calcul du succès : l'audience (Reach) multipliée par la conversion
    ROUND(REACH * CONVERSION_RATE, 0) AS ESTIMATION_NOMBRE_VENTES,
    -- Calcul du Revenu estimé (en multipliant par un panier moyen fictif de 15€ par exemple)
    ROUND((REACH * CONVERSION_RATE) * 15, 2) AS REVENU_ESTIME,
    -- Calcul de l'efficacité (Revenu / Budget)
    ROUND(REVENU_ESTIME / NULLIF(BUDGET, 0), 2) AS RATIO_PERFORMANCE
FROM MARKETING_CAMPAIGNS_CLEAN
WHERE BUDGET > 0
ORDER BY REVENU_ESTIME DESC;

Commentaire : ok (expliquer maintenant)

## Identification des campagnes les plus efficaces

In [ ]:
%%sql -r dataframe_8
USE SCHEMA ANYCOMPANY_LAB.SILVER;

SELECT 
    CAMPAIGN_NAME,
    CAMPAIGN_TYPE,
    BUDGET,
    -- Calcul de l'efficacité relative (Conversion par rapport au Budget)
    -- On multiplie par 100 pour avoir un score lisible
    ROUND((REACH * CONVERSION_RATE) / NULLIF(BUDGET, 0), 4) AS EFFICIENCY_SCORE,
    -- Calcul du coût par client acquis (CPA)
    ROUND(BUDGET / NULLIF((REACH * CONVERSION_RATE), 0), 2) AS COST_PER_CONVERSION,
    -- Performance brute
    ROUND(REACH * CONVERSION_RATE, 0) AS ESTIMATED_CONVERSIONS
FROM MARKETING_CAMPAIGNS_CLEAN
WHERE BUDGET > 0 
  AND ESTIMATED_CONVERSIONS > 0
ORDER BY EFFICIENCY_SCORE DESC;

Commentaire : ok (expliquer maintenant)

# 3- Expérience client
- Impact des avis produits sur les ventes

In [ ]:
%%sql -r dataframe_9
USE SCHEMA ANYCOMPANY_LAB.SILVER;

SELECT 
    PRODUCT_ID,
    ROUND(AVG(RATING), 2) AS NOTE_MOYENNE,
    COUNT(REVIEW_ID) AS NOMBRE_AVIS,
    -- On identifie les produits critiques (note < 3)
    CASE 
        WHEN AVG(RATING) >= 4 THEN 'Excellent (Top Sales Potential)'
        WHEN AVG(RATING) >= 3 THEN 'Moyen'
        ELSE 'Critique (Risque de baisse de ventes)'
    END AS STATUT_SATISFACTION
FROM PRODUCT_REVIEWS_CLEAN
GROUP BY 1
ORDER BY NOTE_MOYENNE DESC;

Commentaire : ok (expliquer)

## Influence des interactions service client

In [ ]:
%%sql -r dataframe_11
USE SCHEMA ANYCOMPANY_LAB.SILVER;

SELECT 
    ISSUE_CATEGORY,
    RESOLUTION_STATUS,
    COUNT(INTERACTION_ID) AS NOMBRE_CAS,
    -- Arrondi à 1 chiffre après la virgule
    ROUND(AVG(CUSTOMER_SATISFACTION), 1) AS SATISFACTION_MOYENNE,
    -- Qualification qualitative simple (sans les intervalles écrits)
    CASE 
        WHEN AVG(CUSTOMER_SATISFACTION) >= 4.5 THEN 'Exceptionnel'
        WHEN AVG(CUSTOMER_SATISFACTION) >= 3.8 THEN 'Satisfaisant'
        WHEN AVG(CUSTOMER_SATISFACTION) >= 3.0 THEN 'Moyennement satisfait'
        WHEN AVG(CUSTOMER_SATISFACTION) >= 2.0 THEN 'Insatisfaisant'
        ELSE 'Critique'
    END AS APPRECIATION_CLIENT
FROM CUSTOMER_SERVICE_INTERACTIONS_CLEAN
GROUP BY 1, 2
ORDER BY NOMBRE_CAS DESC;

Commentaire : il faut modifier la ligne 7 pour obtenir un interval et qualifier l'interval (excellent, bien, assez-bien...) ok

# 4- Opérations et logistique
- Ruptures de stock

In [ ]:
%%sql -r dataframe_12
USE SCHEMA ANYCOMPANY_LAB.SILVER;

SELECT 
    I.PRODUCT_ID,
    I.PRODUCT_CATEGORY,
    I.COUNTRY, -- Ajout de la colonne pays
    I.CURRENT_STOCK,
    -- 1. Création des alertes par seuils
    CASE 
        WHEN I.CURRENT_STOCK < 500 THEN '🔴 Rupture imminente'
        WHEN I.CURRENT_STOCK >= 500 AND I.CURRENT_STOCK < 1000 THEN '🟠 Stock faible'
        WHEN I.CURRENT_STOCK >= 1000 AND I.CURRENT_STOCK < 1500 THEN '🟡 Stock bas'
        WHEN I.CURRENT_STOCK >= 1500 AND I.CURRENT_STOCK < 2500 THEN '🟢 Stock correct'
        ELSE '🔵 Stock optimal'
    END AS STATUT_ALERTE,
    -- 2. Priorité de réapprovisionnement (basée sur le stock seul ici)
    CASE 
        WHEN I.CURRENT_STOCK < 1000 THEN 'À réapproisionner'
        ELSE 'À surveiller'
    END AS PRIORITE_ACTION
FROM INVENTORY_CLEAN I
GROUP BY 1, 2, 3, 4 -- Mise à jour des index pour inclure le pays et le stock
ORDER BY I.CURRENT_STOCK ASC;

Commentaire : créer des cases avec des colonnes alertes pour un seuil de stock bas (par exemple si le niveau de stock<(600) critique, stock<.....) évaluer également en fonction du niveau des produits les plus demandé. OK

## Impact des délais de livraison


In [ ]:
%%sql -r dataframe_14
USE SCHEMA ANYCOMPANY_LAB.SILVER;

WITH BASE_STATS AS (
    SELECT 
        L.DESTINATION_COUNTRY AS PAYS,
        I.PRODUCT_CATEGORY,
        ROUND(AVG(DATEDIFF('day', L.SHIP_DATE, L.ESTIMATED_DELIVERY)), 1) AS DELAI_MOYEN,
        COUNT(L.SHIPMENT_ID) AS VOLUME_COMMANDES,
        ROUND((AVG(DATEDIFF('day', L.SHIP_DATE, L.ESTIMATED_DELIVERY)) * COUNT(L.SHIPMENT_ID)) / 100, 2) AS SCORE_IMPACT
    FROM LOGISTICS_AND_SHIPPING_CLEAN L
    INNER JOIN INVENTORY_CLEAN I ON L.DESTINATION_COUNTRY = I.COUNTRY
    WHERE L.ESTIMATED_DELIVERY IS NOT NULL
    GROUP BY 1, 2
    HAVING VOLUME_COMMANDES > 10
)

SELECT 
    PAYS,
    PRODUCT_CATEGORY,
    DELAI_MOYEN,
    VOLUME_COMMANDES,
    SCORE_IMPACT AS SCORE_IMPACT_LOGISTIQUE,
    
    CASE 
        WHEN SCORE_IMPACT > 50 THEN 'CRITIQUE'
        WHEN SCORE_IMPACT > 20 THEN 'ELEVE'
        WHEN SCORE_IMPACT > 10 THEN 'MODERE'
        ELSE 'FAIBLE'
    END AS NIVEAU_IMPACT,

    CASE 
        WHEN SCORE_IMPACT > 50 THEN 'Changement transporteur ou entrepot local requis'
        WHEN SCORE_IMPACT > 20 THEN 'Optimisation des routes de livraison'
        WHEN SCORE_IMPACT > 10 THEN 'Surveillance des delais'
        ELSE 'Maintenir les operations'
    END AS ACTION_RECO -- J'ai supprimé "RECOMMANDÉE" ici pour corriger l'erreur
FROM BASE_STATS
ORDER BY SCORE_IMPACT DESC;

Commentaire : Il faut resortire l'impact réelle des délais de livraisons sur les commandes. ok